# Compare SED outputs from different runs

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pickle as pkl
import pandas as pd
import prospect.io.read_results as reader
from prospect.utils.plotting import get_percentiles, get_best
from corner import quantile
import h5py

import matplotlib as mpl

#mpl.rcParams["font.family"] = "serif"  # override bagpipes' Helvetica request
mpl.rcParams["text.usetex"] = True

from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import WMAP9 as cosmo
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.sources import FastStepBasis
from astropy.cosmology import Planck18 as cosmo
from prospect.models.sedmodel import PolySpecModel, SpecModel

import fitutils as fit


## Prospector Loading Function

In [ ]:
phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

data = fit.load_prospector_results(12717, prosp_dir)

## Bagpipes Loading Function

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

example_id = 7102

example_res = fit.load_bagpipes_results(21424, bagp_dir)
print(example_res)


## Load results and store them in pickle files

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/fesc_with_miri/pickles'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/fesc_with_miri/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        p_data = fit.load_prospector_results(gal_id, prosp_dir)
        b_data = fit.load_bagpipes_results(gal_id, bagp_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'prospector': p_data,
        'bagpipes': b_data
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

## Check double-peaks

In [ ]:
pickle_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'b': [], 'p': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists

tolerances = {
    'logmass': 0.2,       # 0.2 dex is reasonable
    'logzsol': 0.3,       # 0.3 dex allows for template differences
    'dust2': 0.5,         # Av is often degenerate, so give it more slack
    'duste_gamma': 0.1,
    'dust_index': 0.2,
    'duste_qpah': 1.5,    # Notice this is large because your plot range is 0-10
    'duste_umin': 4.0,    # Large range, large tolerance
    'gas_logu': 0.4
}

file = files[0]

results = {}

file = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/pickles/21218_comp.pkl"
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        objid = data["id"]
        bagp = data['bagpipes']['params']
        prosp = data['prospector']['params']
        
        b_weights = data['bagpipes']['meta']['weights']
        p_weights = data['prospector']['meta']['weights']
        
        results[objid] = {}
        
        for label, tolerance in tolerances.items():
            bag_peaks = fit.find_modes(bagp[label]["samples"], weights=b_weights)
            pros_peaks = fit.find_modes(prosp[label]["samples"], weights=p_weights)
            
            bag_best = bagp[label]["q50"]
            pros_best = prosp[label]["map"]
            
            if len(bag_peaks) == 0:
                gap_bag = 0.0
            else:
                gap_bag = np.abs(bag_peaks[0] - bag_best)    
            if len(pros_peaks) == 0:
                pros_bag = 0.0
            else:   
                gap_pros = np.abs(pros_peaks[0] - pros_best)
            
            if gap_bag > tolerance:
                #print(f"Parameter {label}: Bagpipes median is lies {gap_bag} from largest peak.")
                bag_offset = True
            else:
                bag_offset = False
            if gap_pros > tolerance:
                #print(f"Parameter {label}: Prospector MAP is lies {gap_pros} from largest peak.")
                pros_offset = True
            else:
                pros_offset = False
            
            # DIVIDE GALAXIES INTO CATEGORIES
            overlaps = fit.check_for_overlap(bag_peaks, pros_peaks, tolerance=tolerance)
            #print(f"{label}: {overlaps}")
                
            # Correct QC Flag Assignment
            if (len(overlaps) > 0) and not (bag_offset or pros_offset):
                qc_flag = 1
            elif (len(overlaps) == 0) and not (bag_offset or pros_offset):
                qc_flag = 2
            elif (len(overlaps) > 0) and (bag_offset or pros_offset):
                qc_flag = 3
            else:
                qc_flag = 4
                    
            # Store into the sub-dictionary
            results[objid][label] = {
                "qc_flag": qc_flag, 
                "bag_offset": gap_bag,
                "pros_offset": gap_pros,
                "has_overlap": len(overlaps) > 0
            }
        
import pandas as pd
df = pd.DataFrame.from_dict({(i, j): results[i][j] 
                           for i in results.keys() 
                           for j in results[i].keys()}, orient='index')
        
print(df)
        
        

Save the dataframe for future analysis

In [ ]:
df_new = df.reset_index()
#df_new.rename(columns={'index': 'id'}, inplace=True)
df_new.rename(columns={"level_0": "id", "level_1": "param", "category": "qc_flag"}, inplace=True)

df_new.to_csv('/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.csv', index=False)

Analyse the dataframe now

In [ ]:
import pandas as pd

# 1. Load the data
df3 = pd.read_csv('/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.csv', index_col=0)

# Convert back to dictionary
results = {}
for objid, row in df3.iterrows():
    if objid not in results:
        results[objid] = {}
    
    label = row['param']
    qc_flag = row['qc_flag']
    gap_bag = row["bag_offset"]
    gap_pros = row["pros_offset"]
    has_overlaps = row["has_overlap"]
    
    results[objid][label] = {
        "qc_flag": qc_flag, 
        "bag_offset": gap_bag,
        "pros_offset": gap_pros,
        "has_overlap": len(overlaps) > 0
    }

# Got it, now this rebuilt my dictionary like I saved it!
print(results)

Plotting the `qc_flag` param

In [ ]:
from matplotlib.figure import SubFigure
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert the dict to a DataFrame
# This flattens the dictionary: results[objid][label]['qc_flag']
#data_for_heatmap = {
#    objid: {label: values['qc_flag'] for label, values in labels.items()}
#    for objid, labels in results.items()
#}

data_for_heatmap = {}
for objid, labels in results.items():
    data_for_heatmap[objid] = {}
    for label, values in labels.items():
        data_for_heatmap[objid][label] = values['qc_flag']

df_heatmap = pd.DataFrame.from_dict(data_for_heatmap, orient='index')

# Sort rows by index (Galaxy ID) ascending
df_heatmap = df_heatmap.sort_index()

# 2. Setup the Heatmap
fig, ax = plt.subplots(figsize=(5, 20))

# Use a discrete color map (1=Green, 2=Yellow, 3=Orange, 4=Red)
cmap = sns.color_palette(["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], as_cmap=True)

sns.heatmap(df_heatmap, 
            cbar=False,
            cmap=cmap, 
            annot=True, 
            #cbar_kws={'ticks': [1, 2, 3, 4]}, 
            vmin=0.5, vmax=4.5)

# Calculate stats for the labels
means = df_heatmap.mean()
medians = df_heatmap.median()

# Create a copy to add summary rows
df_plot = df_heatmap.copy()
df_plot.loc['MEAN'] = means
df_plot.loc['MEDIAN'] = medians

# Create labels with stats (e.g., "logmass\nμ=1.2, md=1.0")
#labels = [f"{col}\nμ={m:.2f}\nmed={med:.1f}" for col, m, med in zip(df_heatmap.columns, means, medians)]

# Move the x-axis ticks to the top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

ax.set_xticks(np.arange(df_heatmap.shape[1])+0.5)
ax.set_xticklabels(labels, rotation=45, ha='left', fontsize=12)

legend = "Quality flagging criteria:\n"
legend += "1: Peaks agree AND no offset (Peak - MAP)\n"
legend += "2: Peaks don't agree AND no offset (Peak - MAP)\n"
legend += "3: Peaks agree AND significant offset (Peak - MAP)\n"
legend += "4: Peaks don't agree AND significant offset (Peak - MAP)\n\n"

labels = ""
for col, m, med in zip(df_heatmap.columns, means, medians):
    labels += f"{col}: μ={m:.2f}, med={med:.1f}\n"

stats_legend = legend + labels

plt.figtext(0.95, 0.8, stats_legend, verticalalignment='center')
#plt.subplots_adjust(right=0.8) # Make room for text on the right
#ax.text(x=1.0, y=0.5, s=stats_legend)
# Clean up axes
#ax.set_xlabel("")

plt.title("Posterior Agreement - Bagpipes vs. Prospector", fontsize=14)
#plt.xlabel("Parameter")
#plt.ylabel("Galaxy ID")
plt.tight_layout()
fig_path = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/no_fesc_with_miri/peak_analysis.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

## Plot parameter comparison Bagpipes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats
import glob
import os

pickle_dir = './comparison/no_fesc_with_miri/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'b': [], 'p': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        bagp = data['bagpipes']['params']
        prosp = data['prospector']['params']
        
        for p_key in collected_data.keys():
            if p_key in bagp and p_key in prosp:
                # Store the MAP value for this galaxy
                collected_data[p_key]['b'].append(bagp[p_key]['q50'])
                collected_data[p_key]['p'].append(prosp[p_key]['map'])

fig_path = './comparison/no_fesc_with_miri/withorwithout_best.png'

# Define the parameters we want to plot
# Format: (conceptual_name, df_prefix, display_label)
params = [
    ('logmass', r'$\log_{10}(M_*/M_\odot)$'),
    ('logzsol', r'$\log_{10}(Z/Z_\odot)$'),
    ('dust2', r'$A_V$ [mag]'),
    ('gas_logu', r'$\log_{10}(U)$'),
    ('duste_qpah', r'$q_{PAH}$ [%]'),
    ('duste_umin', r'Dust $U_{min}$'),
    ('duste_gamma', r'Dust $\gamma$'),
    ('dust_index', r'$n_{dust}$')
]

n_params = len(params)
fig, axes = plt.subplots(2, n_params//2, figsize=(14, 8))

axes = axes.flatten()

for i, (col, label) in enumerate(params):
    ax = axes[i]
    
    x = np.array(collected_data[col]['b'])
    y = np.array(collected_data[col]['p'])
    
    if len(x) == 0: continue

    # Calculate limits
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    pad = (vmax - vmin) * 0.1
    vmin, vmax = vmin - pad, vmax + pad

    # Plot 1:1 Line
    ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)
    
    # Labels and formatting
    ax.set_title(f'{label}', fontsize=14)
    ax.set_xlabel(f'Bagpipes', fontsize=12)
    ax.set_ylabel(f'Prospector', fontsize=12)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Calculate and display Mean Offset
    if i in [0, 2]:
        offset = np.nanmedian(y - x)
        scatter = median_abs_deviation(y - x)
        ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    else:
        corr_coef, p_value = stats.pearsonr(x, y)
        ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


## Create corner plots for the whole sample!

In [ ]:
comp_dir = './comparison/fesc_with_miri/'

pickle_files = glob.glob(os.path.join(comp_dir, 'pickles', '*_comp.pkl'))

for p in pickle_files:
    objid = int(os.path.basename(p).split('_comp.pkl')[0])  # Extract the galaxy ID from the filename
    fit.plot_dual_corner(objid, comp_dir)
    print(f"Plotted corner plot for galaxy ID: {objid}")

## Individual Bagpipes output

In [ ]:
objid=21424

bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/fesc_with_miri/'

example_res = fit.load_bagpipes_results(21424, bagp_dir)

fit.plot_corner(galaxy_id=objid, data=example_res, title=f"Galaxy {objid} Bagpipes", color="Orange")

## Individual Prospector Output

In [ ]:
objid=21424

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

example_res = fit.load_prospector_results(objid, prosp_dir)

fit.plot_corner(galaxy_id=objid, data=example_res, title=f"Galaxy {objid} Prospector", color="dodgerblue")

# Compare Prospector with and without MIRI

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/prospector/pickles'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

miri_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
no_miri_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.0.2/sourcephotonly_v2.0.2/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        data1 = load_prospector_results(gal_id, miri_dir)
        data2 = load_prospector_results(gal_id, no_miri_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'miri': data1,
        'no_miri': data2
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

Plot the parameters before and after

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats

pickle_dir = './comparison/prospector/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

# Initialise a dictionary to collect MAP values for ALL galaxies
collected_data = {p[0]: {'with': [], 'without': []} for p in [
    ('logmass', ''), ('logzsol', ''), ('dust2', ''), ('gas_logu', ''),
    ('duste_qpah', ''), ('duste_umin', ''), ('duste_gamma', ''), ('dust_index', '')
]}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        # Accessing your structure: data['miri'] and data['no_miri'] are dictionaries
        miri = data['miri']['params']
        no_miri = data['no_miri']['params']
        
        for p_key in collected_data.keys():
            if p_key in miri and p_key in no_miri:
                # Store the MAP value for this galaxy
                collected_data[p_key]['with'].append(miri[p_key]['q50'])
                collected_data[p_key]['without'].append(no_miri[p_key]['q50'])

fig_path = './comparison/prospector/plots/withorwithout_median.png'

# Define the parameters we want to plot
# Format: (conceptual_name, df_prefix, display_label)
params = [
    ('logmass', r'$\log_{10}(M_*/M_\odot)$'),
    ('logzsol', r'$\log_{10}(Z/Z_\odot)$'),
    ('dust2', r'$A_V$ [mag]'),
    ('gas_logu', r'$\log_{10}(U)$'),
    ('duste_qpah', r'$q_{PAH}$ [%]'),
    ('duste_umin', r'Dust $U_{min}$'),
    ('duste_gamma', r'Dust $\gamma$'),
    ('dust_index', r'$n_{dust}$')
]

n_params = len(params)
fig, axes = plt.subplots(2, n_params//2, figsize=(14, 8))

axes = axes.flatten()

for i, (col, label) in enumerate(params):
    ax = axes[i]
    
    x = np.array(collected_data[col]['with'])
    y = np.array(collected_data[col]['without'])
    
    if len(x) == 0: continue

    # Calculate limits
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    pad = (vmax - vmin) * 0.1
    vmin, vmax = vmin - pad, vmax + pad

    # Plot 1:1 Line
    ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)
    
    # Labels and formatting
    ax.set_title(f'{label}', fontsize=14)
    ax.set_xlabel(f'With MIRI', fontsize=12)
    ax.set_ylabel(f'No MIRI', fontsize=12)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Calculate and display Mean Offset
    if i in [0, 2]:
        offset = np.nanmedian(y - x)
        scatter = median_abs_deviation(y - x)
        ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    else:
        corr_coef, p_value = stats.pearsonr(x, y)
        ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


## Dust2 with and without MIRI

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import scipy.stats as stats
import glob
import os

pickle_dir = './comparison/prospector/pickles'
files = glob.glob(os.path.join(pickle_dir, '*_comp.pkl'))

dust2 = {'id': [], 'with': [], 'without': []}

# 2. Loop through files and fill the lists
for file in files:
    with open(file, 'rb') as f:
        data = pkl.load(f)
        dust2_miri = data['miri']['params']['dust2']['q50']*1.086
        dust2_no_miri = data['no_miri']['params']['dust2']['q50']*1.086
        
        dust2['id'].append(data['id'])
        dust2['with'].append(dust2_miri)
        dust2['without'].append(dust2_no_miri)
        
        if data['id'] == 7549:
            print(f"Galaxy {data['id']} - Dust2 with MIRI: {dust2_miri:.3f}, without MIRI: {dust2_no_miri:.3f}")

N = len(dust2['with'])
k = int(1 + np.log2(N))*2

range = [min(min(dust2['with']), min(dust2['without'])), max(max(dust2['with']), max(dust2['without']))]

fig, ax = plt.subplots(figsize=(5, 5))
ax.hist(dust2['with'], bins=k, alpha=0.5, label='With MIRI', color='dodgerblue', range=range)#, edgecolor='black')
ax.hist(dust2['without'], bins=k, alpha=0.5, label='Without MIRI', color='coral', range=range)#, edgecolor='black')
#ax.vlines(np.median(dust2['with']), ymin=0, ymax=ax.get_ylim()[1], color='dodgerblue', linestyle='--', label='Median With MIRI')
#ax.vlines(np.median(dust2['without']), ymin=0, ymax=ax.get_ylim()[1], color='coral', linestyle='--', label='Median Without MIRI')
ax.set_xlabel(r'$A_V$', fontsize=14)
ax.set_ylabel('Frequency', fontsize=14)
ax.tick_params(labelsize=14)
ax.legend()
plt.title(r'$A_V$ with and without MIRI', fontsize=16)
plt.show()

Now let's look at the difference

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

deltas = np.array(dust2['with']) - np.array(dust2['without'])   # Difference in dust2 after adding MIRI
imax = np.argmax(deltas)
print(imax)
print(deltas[imax])
print(list(dust2['id'])[imax])

ratio = np.array(dust2['with']) / np.array(dust2['without'])

print(np.median(ratio))

ax.scatter(dust2['without'], deltas, alpha=0.7, color='dodgerblue', edgecolor='black', s=50)
ax.hlines(0, xmin=0, xmax=ax.get_xlim()[1], color='gray', linestyle='--', alpha=0.7, linewidth=2.5)
ax.set_ylim(-2.0, 2.0)
ax.set_xlabel(r'$A_V$ without MIRI', fontsize=14)
ax.set_ylabel(r'$\Delta$ ', fontsize=14)
ax.tick_params(labelsize=14)
plt.title(r'$\Delta$ = $A_V$ with MIRI - $A_V$ without MIRI', fontsize=16)
plt.savefig('./comparison/prospector/plots/dust2_delta_median.png', dpi=300, bbox_inches='tight')
plt.show()

Corner Plot for ID 7549

In [ ]:
objid=10339
pickle_file = f'./comparison/prospector/pickles/{objid}_comp.pkl'

# 2. Loop through files and fill the lists
with open(pickle_file, 'rb') as f:
    data = pkl.load(f)
    with_miri = data['miri']
    no_miri = data['no_miri']
    
    print(f"Galaxy {data['id']} - Dust2 with MIRI: {with_miri['params']['dust2']['map']*1.086:.3f}, without MIRI: {no_miri['params']['dust2']['map']*1.086:.3f}")
    
fig = fit.plot_dual_corner(galaxy_id=objid,
                           data1=no_miri, data2=with_miri, 
                           label1="No MIRI", label2="With MIRI", 
                           save_fig=False,
                           title=f"Galaxy {objid}")

# Mock galaxy fits

New corner plot function

In [ ]:
objid = 9997

comp_dir = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/mock_fit"

p_dir = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/prospector/output"
b_dir = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/mock_fit"

p_data = fit.load_prospector_results(objid, p_dir)
b_data = fit.load_bagpipes_results(objid, b_dir)

mock_params = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/mock_fit/{objid}_mock.npz"
mock_data = np.load(mock_params, allow_pickle=True)

fit.plot_dual_corner(galaxy_id=objid, 
                 data1=b_data, label1="Bagpipes",
                 data2=p_data, label2="Prospector", 
                 mock_data=mock_data,
                 save_dir=comp_dir,
                 title=f"Bagpipes vs. Prospector\nMock Galaxy {objid}",
                 scale_dust2=True,   # for Prospector
                 save_fig=False
                 )

Create SFH from Prospector

In [ ]:
objid=9997
output = glob.glob(f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison/prospector/output/{objid}*_mcmc.h5")
h5_file = output[0]

fit.plot_sfh_prospector(h5_file, figname=f"{objid}_sfh_prospector.pdf")